In [ ]:
#EXTREME OUTLIER ANALYSIS
"""
This script is deisgned to exclude statistically unlikely results based on the extremely high or low z-scores. 
Values are excluded if they are > 3 SD from mean without any other tests equalling |1.5| from the mean in the same instance 
It firsts runs an inflation analysis to see the extent of inflated tails in each test
Extreme vlaues are then visualised overlayed on the per-test z-score distribution
Descriptive stats for each cleaned test are then given
"""

import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load data
df = pd.read_csv("cv_results_all_tests.csv")

# Define reverse-scored tests
reverse_tests = ["PMat", "RT", "TMA", "TMB"]

thresholds = [2, 2.5, 3, 3.5, 4]
rows = []


In [ ]:
#Extreme impairment results inflation analysis

for test, subdf in df.groupby("test_name"):
    n = len(subdf)
    for t in thresholds:
        # Identify impairment-only extremes for this test
        if test in reverse_tests:
            obs_count = (subdf["z_score"] > t).sum()
        else:
            obs_count = (subdf["z_score"] < -t).sum()

        obs_percent = 100 * obs_count / n if n > 0 else 0

        # Expected one-tailed percent
        exp_percent = 100 * (1 - stats.norm.cdf(t))
        exp_count = n * exp_percent / 100

        rows.append({
            "test_name": test,
            "threshold": t,
            "n_total": n,
            "observed_count": obs_count,
            "observed_percent": obs_percent,
            "expected_percent": exp_percent,
            "expected_count": exp_count,
            "inflation_ratio": obs_percent / exp_percent if exp_percent > 0 else None
        })

per_test_impairment = pd.DataFrame(rows)

per_test_impairment.to_csv("inflation_per_test_imp.csv", index=False)
display(per_test_impairment)

In [ ]:
# Function to determine extreme impairment (>3 SD worse)
def is_extreme_imp(row):
    if row["test_name"] in reverse_tests:
        return row["z_score"] > 3
    else:
        return row["z_score"] < -3

# Function to determine if performance is acceptable (within 1.5 SD)
def is_acceptable(row):
    if row["test_name"] in reverse_tests:
        return row["z_score"] < 1.5
    else:
        return row["z_score"] > -1.5

# Add flags for extreme and acceptable performance
df["extreme"] = df.apply(is_extreme_imp, axis=1)
df["acceptable"] = df.apply(is_acceptable, axis=1)

# Identify participants with at least one extreme impairment
extreme_imp_df = df[df["extreme"]].copy()
participants_with_extreme = extreme_imp_df["eid"].unique()

# For participants with extreme impairments, check if others are acceptable
isolated_participants = []

for eid in participants_with_extreme:
    participant_data = df[df["eid"] == eid]
    
    # Check if participant has extreme impairments AND all other tests are acceptable
    has_extreme = participant_data["extreme"].any()
    non_extreme_tests = participant_data[~participant_data["extreme"]]
    all_others_acceptable = non_extreme_tests["acceptable"].all() if len(non_extreme_tests) > 0 else True
    
    if has_extreme and all_others_acceptable:
        isolated_participants.append(eid)

# Create summary with refined isolated definition
participant_summary = []

for eid in participants_with_extreme:
    participant_data = df[df["eid"] == eid]
    extreme_count = participant_data["extreme"].sum()
    
    if eid in isolated_participants:
        status = "isolated"
    else:
        status = "multiple"
    
    participant_summary.append({
        "eid": eid,
        "extreme_imp_count": extreme_count,
        "status": status
    })

counts_imp = pd.DataFrame(participant_summary)

# Extract isolated impairments
isolated_imp_ids = counts_imp.loc[counts_imp["status"] == "isolated", "eid"]
isolated_imp_df = extreme_imp_df[extreme_imp_df["eid"].isin(isolated_imp_ids)]


# Overall comparison
overall_comparison_imp = counts_imp["status"].value_counts().reset_index()
overall_comparison_imp.columns = ["status", "count"]
overall_comparison_imp["percent"] = 100 * overall_comparison_imp["count"] / overall_comparison_imp["count"].sum()
overall_comparison_imp.to_csv("isolated_vs_multiple_impairment.csv", index=False)

# Per-test summary of isolated impairments
isolated_imp_summary = (
    isolated_imp_df.groupby("test_name")
    .size()
    .reset_index(name="isolated_extreme_imp_count")
    .merge(
        df.groupby("test_name").size().reset_index(name="total_instances"),
        on="test_name"
    )
)
# Add total extreme count per test
total_extreme_per_test = extreme_imp_df.groupby("test_name").size().reset_index(name="total_extreme_count")
isolated_imp_summary = isolated_imp_summary.merge(total_extreme_per_test, on="test_name")

# Calculate percentages
isolated_imp_summary["isolated_extreme_imp_percent"] = (
    100 * isolated_imp_summary["isolated_extreme_imp_count"] / isolated_imp_summary["total_instances"]
)
isolated_imp_summary["isolated_of_extreme_percent"] = (
    100 * isolated_imp_summary["isolated_extreme_imp_count"] / isolated_imp_summary["total_extreme_count"]
)

# Add descriptive stats
stats = df.groupby("test_name")["z_score"].agg(["mean", "std"]).reset_index()
isolated_imp_summary = isolated_imp_summary.merge(stats, on="test_name")


print("\nIsolated impairment definition:")
print("- At least one test >3 SD worse than expected")
print("- All other tests within 1.5 SD of expected performance")

print("\nDisplaying results:")
print(f"\nTotal participants with extreme impairments: {len(participants_with_extreme)}")
print(f"Participants with isolated impairments: {len(isolated_participants)}")
print(f"Participants with multiple/complex impairments: {len(participants_with_extreme) - len(isolated_participants)}")

display(overall_comparison_imp)
display(isolated_imp_summary)
display(isolated_imp_df)  # Show first 10 for brevity

# Save files
extreme_imp_df.to_csv("extreme_impairments.csv", index=False)
counts_imp.to_csv("impairment_summary.csv", index=False)
isolated_imp_df.to_csv("isolated_impairments.csv", index=False)
isolated_imp_summary.to_csv("isolated_impairment_extremes_by_test.csv", index=False)

print("""Tables saved: 
        
        extreme_impairments.csv
        impairment_summary.csv 
        isolated_impairments.csv
        isolated_vs_multiple_impairment.csv
        isolated_impairment_extremes_by_test.csv""")

In [ ]:
# Extreme results inflation analysis - over-performers

import scipy.stats as stats

for test, subdf in df.groupby("test_name"):
    n = len(subdf)
    for t in thresholds:
        # Identify overachievement extremes for this test
        if test in reverse_tests:
            obs_count = (subdf["z_score"] < -t).sum()
        else:
            obs_count = (subdf["z_score"] > t).sum()

        obs_percent = 100 * obs_count / n if n > 0 else 0

        # Expected one-tailed percent
        exp_percent = 100 * (1 - stats.norm.cdf(t))
        exp_count = n * exp_percent / 100

        rows.append({
            "test_name": test,
            "threshold": t,
            "n_total": n,
            "observed_count": obs_count,
            "observed_percent": obs_percent,
            "expected_percent": exp_percent,
            "expected_count": exp_count,
            "inflation_ratio": obs_percent / exp_percent if exp_percent > 0 else None
        })

per_test_overachievers = pd.DataFrame(rows)

# Save + show
print("Results saved to per_test_overachievers_extremes.csv")
per_test_overachievers.to_csv("inflation_per_test_ovr.csv", index=False)
display(per_test_overachievers)

In [ ]:
# Function to determine extreme overachievement (>3 SD better)
def is_extreme_over(row):
    if row["test_name"] in reverse_tests:
        return row["z_score"] < -3
    else:
        return row["z_score"] > 3
# Function to determine if performance is acceptable (within 1.5 SD)
def is_acceptable(row):
    if row["test_name"] in reverse_tests:
        return row["z_score"] > -1.5
    else:
        return row["z_score"] < 1.5

# Add flags for extreme and acceptable performance
df["extreme"] = df.apply(is_extreme_over, axis=1)
df["acceptable"] = df.apply(is_acceptable, axis=1)

# Identify participants with at least one extreme overachievement
extreme_over_df = df[df["extreme"]].copy()
participants_with_extreme = extreme_over_df["eid"].unique()

# For participants with extreme overachievements, check if others are acceptable
isolated_participants = []
for eid in participants_with_extreme:
    participant_data = df[df["eid"] == eid]
    
    # Check if participant has extreme overachievements AND all other tests are acceptable
    has_extreme = participant_data["extreme"].any()
    non_extreme_tests = participant_data[~participant_data["extreme"]]
    all_others_acceptable = non_extreme_tests["acceptable"].all() if len(non_extreme_tests) > 0 else True
    
    if has_extreme and all_others_acceptable:
        isolated_participants.append(eid)

# Create summary with refined isolated definition
participant_summary = []
for eid in participants_with_extreme:
    participant_data = df[df["eid"] == eid]
    extreme_count = participant_data["extreme"].sum()
    
    if eid in isolated_participants:
        status = "isolated"
    else:
        status = "multiple"
    
    participant_summary.append({
        "eid": eid,
        "extreme_over_count": extreme_count,
        "status": status
    })
counts_over = pd.DataFrame(participant_summary)

# Extract isolated overachievements
isolated_over_ids = counts_over.loc[counts_over["status"] == "isolated", "eid"]
isolated_over_df = extreme_over_df[extreme_over_df["eid"].isin(isolated_over_ids)]

# Overall comparison
overall_comparison_over = counts_over["status"].value_counts().reset_index()
overall_comparison_over.columns = ["status", "count"]
overall_comparison_over["percent"] = 100 * overall_comparison_over["count"] / overall_comparison_over["count"].sum()
overall_comparison_over.to_csv("isolated_vs_multiple_overachievement.csv", index=False)

# Per-test summary of isolated overachievements
isolated_over_summary = (
    isolated_over_df.groupby("test_name")
    .size()
    .reset_index(name="isolated_extreme_over_count")
    .merge(
        df.groupby("test_name").size().reset_index(name="total_instances"),
        on="test_name"
    )
)
# Add total extreme count per test
total_extreme_per_test = extreme_over_df.groupby("test_name").size().reset_index(name="total_extreme_count")
isolated_over_summary = isolated_over_summary.merge(total_extreme_per_test, on="test_name")

# Calculate percentages
isolated_over_summary["isolated_extreme_over_percent"] = (
    100 * isolated_over_summary["isolated_extreme_over_count"] / isolated_over_summary["total_instances"]
)
isolated_over_summary["isolated_of_extreme_percent"] = (
    100 * isolated_over_summary["isolated_extreme_over_count"] / isolated_over_summary["total_extreme_count"]
)

# Add descriptive stats
stats = df.groupby("test_name")["z_score"].agg(["mean", "std"]).reset_index()
isolated_over_summary = isolated_over_summary.merge(stats, on="test_name")
print("\nIsolated overachievement definition:")
print("- At least one test >3 SD better than expected")
print("- All other tests within 1.5 SD of expected performance")
print("\nDisplaying results:")
print(f"\nTotal participants with extreme overachievements: {len(participants_with_extreme)}")
print(f"Participants with isolated overachievements: {len(isolated_participants)}")
print(f"Participants with multiple/complex overachievements: {len(participants_with_extreme) - len(isolated_participants)}")
display(overall_comparison_over)
display(isolated_over_summary)
display(isolated_over_df)  # Show first 10 for brevity

# Save files
extreme_over_df.to_csv("extreme_overachievements.csv", index=False)
counts_over.to_csv("overachievement_summary.csv", index=False)
isolated_over_df.to_csv("isolated_overachievements.csv", index=False)
isolated_over_summary.to_csv("isolated_overachievement_extremes_by_test.csv", index=False)
print("""Tables saved: 
        
        extreme_overachievements.csv
        overachievement_summary.csv 
        isolated_overachievements.csv
        isolated_vs_multiple_overachievement.csv
        isolated_overachievement_extremes_by_test.csv""")

In [ ]:
#Now remove the extreme values

isolated_imp = pd.read_csv("isolated_impairments.csv")
isolated_over = pd.read_csv("isolated_overachievements.csv")

print(f"Original data shape: {df.shape}")
print(f"Isolated impairments to remove: {isolated_imp.shape[0]}")
print(f"Isolated overachievements to remove: {isolated_over.shape[0]}")


# Create summary BEFORE cleaning
summary_before = df.groupby("test_name").size().reset_index(name="instances_before")

# Create a merge key for matching
df['merge_key'] = df['eid'].astype(str) + '_' + df['test_name'] + '_' + df['instance'].astype(str)
isolated_imp['merge_key'] = isolated_imp['eid'].astype(str) + '_' + isolated_imp['test_name'] + '_' + isolated_imp['instance'].astype(str)
isolated_over['merge_key'] = isolated_over['eid'].astype(str) + '_' + isolated_over['test_name'] + '_' + isolated_over['instance'].astype(str)

# Mark rows to remove
df['is_isolated_imp'] = df['merge_key'].isin(isolated_imp['merge_key'])
df['is_isolated_over'] = df['merge_key'].isin(isolated_over['merge_key'])

# Count removals per test
removal_counts = df.groupby('test_name').agg({
    'is_isolated_imp': 'sum',
    'is_isolated_over': 'sum'
}).reset_index()
removal_counts.columns = ['test_name', 'impairments_removed', 'overachievements_removed']
removal_counts['total_removed'] = removal_counts['impairments_removed'] + removal_counts['overachievements_removed']

# Remove the isolated extreme values
df_cleaned = df[~(df['is_isolated_imp'] | df['is_isolated_over'])].copy()

# Drop the helper columns
df_cleaned = df_cleaned.drop(columns=['merge_key', 'is_isolated_imp', 'is_isolated_over'])

print(f"\nCleaned data shape: {df_cleaned.shape}")
print(f"Total rows removed: {df.shape[0] - df_cleaned.shape[0]}")

# Create summary AFTER cleaning
summary_after = df_cleaned.groupby("test_name").size().reset_index(name="instances_after")

# Merge all summaries
summary_table = summary_before.merge(removal_counts, on='test_name', how='left')
summary_table = summary_table.merge(summary_after, on='test_name', how='left')

# Convert counts to integers where they exist (but leave NaN as NaN)
for col in ['impairments_removed', 'overachievements_removed', 'total_removed']:
    summary_table[col] = summary_table[col].astype('Int64')  # Nullable integer type

# Calculate percentage removed (will be NaN if no removals)
summary_table['percent_removed'] = (
    100 * summary_table['total_removed'] / summary_table['instances_before']
).round(2)

# Reorder columns
summary_table = summary_table[[
    'test_name',
    'instances_before',
    'impairments_removed',
    'overachievements_removed',
    'total_removed',
    'instances_after',
    'percent_removed'
]]

print("\n" + "="*80)
print("CLEANING SUMMARY")
print("="*80)
print(summary_table.to_string(index=False))

# Save cleaned data
output_path = "cv_results_all_tests_cleaned.csv"
df_cleaned.to_csv(output_path, index=False)
print(f"\nCleaned data saved to: {output_path}")

# Save summary table
summary_path = "cleaning_summary.csv"
summary_table.to_csv(summary_path, index=False)
print(f"Summary table saved to: {summary_path}")

# Overall summary
print("\n" + "="*80)
print("OVERALL SUMMARY")
print("="*80)
print(f"Total instances before cleaning: {summary_table['instances_before'].sum():,}")
print(f"Total impairments removed: {summary_table['impairments_removed'].sum():,}")
print(f"Total overachievements removed: {summary_table['overachievements_removed'].sum():,}")
print(f"Total instances removed: {summary_table['total_removed'].sum():,}")
print(f"Total instances after cleaning: {summary_table['instances_after'].sum():,}")
print(f"Overall percent removed: {100 * summary_table['total_removed'].sum() / summary_table['instances_before'].sum():.2f}%")

In [ ]:
#Calculate Post-clean statistics

# Load cleaned data
df = pd.read_csv("cv_results_all_tests_cleaned.csv")

print(f"Total observations in cleaned dataset: {len(df):,}")
print(f"Unique participants: {df['eid'].nunique():,}")
print(f"Tests: {', '.join(sorted(df['test_name'].unique()))}")

# Function to calculate statistics per test
def calculate_test_stats(test_df):
    stats = {}
    
    # Basic counts
    stats['n_instances'] = len(test_df)
    stats['n_participants'] = test_df['eid'].nunique()
    
    # Test score statistics
    stats['mean_score'] = test_df['test_score'].mean()
    stats['std_score'] = test_df['test_score'].std()
    stats['median_score'] = test_df['test_score'].median()
    stats['min_score'] = test_df['test_score'].min()
    stats['max_score'] = test_df['test_score'].max()
    
    # Demographics
    stats['mean_age'] = test_df['age_at_test'].mean()
    stats['n_male'] = (test_df['Sex'] == 1).sum()
    stats['n_female'] = (test_df['Sex'] == 0).sum()
    
    # Education - use mode for ordinal scale
    edu_mode = test_df['Edu'].mode()
    stats['mode_edu'] = edu_mode[0] if len(edu_mode) > 0 else np.nan
    
    # Repeated measures analysis
    instance_counts = test_df.groupby('eid')['instance'].nunique()
    stats['n_repeated'] = (instance_counts > 1).sum()
    
    # Calculate age differences for repeated measures
    repeated_eids = instance_counts[instance_counts > 1].index
    age_diffs = []
    
    for eid in repeated_eids:
        participant_data = test_df[test_df['eid'] == eid].sort_values('instance')
        if len(participant_data) > 1:
            ages = participant_data['age_at_test'].values
            diffs = np.diff(ages)
            age_diffs.extend(diffs)
    
    stats['mean_age_diff'] = np.mean(age_diffs) if age_diffs else np.nan
    
    return stats

# Calculate statistics for each test
results = []
tests = sorted(df['test_name'].unique())

for test in tests:
    test_df = df[df['test_name'] == test]
    stats = calculate_test_stats(test_df)
    stats['test_name'] = test
    results.append(stats)

# Calculate overall statistics
overall_stats = calculate_test_stats(df)
overall_stats['test_name'] = 'TOTAL'
results.append(overall_stats)

# Create DataFrame
desc_table = pd.DataFrame(results)

# Reorder columns
desc_table = desc_table[[
    'test_name',
    'n_instances',
    'n_participants',
    'mean_score',
    'std_score',
    'median_score',
    'min_score',
    'max_score',
    'mean_age',
    'n_male',
    'n_female',
    'mode_edu',
    'n_repeated',
    'mean_age_diff'
]]

# Round numeric columns
desc_table['mean_score'] = desc_table['mean_score'].round(2)
desc_table['std_score'] = desc_table['std_score'].round(2)
desc_table['median_score'] = desc_table['median_score'].round(2)
desc_table['min_score'] = desc_table['min_score'].round(2)
desc_table['max_score'] = desc_table['max_score'].round(2)
desc_table['mean_age'] = desc_table['mean_age'].round(1)
desc_table['mean_age_diff'] = desc_table['mean_age_diff'].round(2)
# mode_edu is already an integer, no rounding needed

# Rename columns for better readability
desc_table.columns = [
    'Test',
    'N Instances',
    'N Participants',
    'Mean Score',
    'SD Score',
    'Median Score',
    'Min Score',
    'Max Score',
    'Mean Age',
    'N Male',
    'N Female',
    'Mode Edu',
    'N w/ Repeats',
    'Mean Age Diff'
]

print("\n" + "="*140)
print("DESCRIPTIVE STATISTICS BY TEST (CLEANED DATA)")
print("="*140)
print(desc_table.to_string(index=False))

# Save to CSV
output_path = "descriptive_stats_cleaned.csv"
desc_table.to_csv(output_path, index=False)
print(f"\nDescriptive statistics saved to: {output_path}")

# Additional summary information
print("\n" + "="*140)
print("ADDITIONAL SUMMARY")
print("="*140)
print(f"Total observations: {len(df):,}")
print(f"Unique participants: {df['eid'].nunique():,}")
print(f"Number of tests: {len(tests)}")
print(f"Overall male/female ratio: {(df['Sex'] == 1).sum():,} / {(df['Sex'] == 0).sum():,}")
print(f"Overall percentage with repeated measures: {100 * (df.groupby('eid')['instance'].nunique() > 1).sum() / df['eid'].nunique():.1f}%")

# Show education distribution
print("\nEducation level distribution (overall):")
print(df['Edu'].value_counts().sort_index())

In [ ]:
#Visualise the results

# Custom color scheme
GREY_DARK = '#2d2d2d'
GREY_MED = '#6d6d6d'
GREY_LIGHT = '#b0b0b0'
GREY_VLIGHT = '#e0e0e0'
RED = '#d62728'
BLUE = '#1f77b4'

# Load main data
df = pd.read_csv("cv_results_all_tests_cleaned.csv")

# Load your pre-calculated isolated cases
isolated_imp_df = pd.read_csv("isolated_impairments.csv")
isolated_over_df = pd.read_csv("isolated_overachievements.csv")

# Mark isolated cases in main dataframe
df["isolated_imp"] = False
df["isolated_over"] = False

for idx, row in isolated_imp_df.iterrows():
    mask = (df['eid'] == row['eid']) & (df['test_name'] == row['test_name']) & (df['instance'] == row['instance'])
    df.loc[mask, 'isolated_imp'] = True

for idx, row in isolated_over_df.iterrows():
    mask = (df['eid'] == row['eid']) & (df['test_name'] == row['test_name']) & (df['instance'] == row['instance'])
    df.loc[mask, 'isolated_over'] = True

print(f"{len(isolated_imp_df)} isolated impairment cases")
print(f"{len(isolated_over_df)} isolated overachievement cases")

tests = sorted(df['test_name'].unique())

# INCREASED FIGURE SIZE to accommodate legend outside
fig, ax = plt.subplots(figsize=(20, 10), facecolor='white')

# Create violin plot
parts = ax.violinplot([df[df['test_name'] == test]['z_score'].values for test in tests],
                      positions=range(len(tests)),
                      showmeans=False,
                      showmedians=False,
                      showextrema=False,
                      widths=0.7)

for pc in parts['bodies']:
    pc.set_facecolor(GREY_LIGHT)
    pc.set_edgecolor(GREY_DARK)
    pc.set_alpha(0.6)
    pc.set_linewidth(1.5)

bp = ax.boxplot([df[df['test_name'] == test]['z_score'].values for test in tests],
                positions=range(len(tests)),
                widths=0.15,
                patch_artist=True,
                showfliers=False,
                medianprops=dict(color='black', linewidth=2.5),
                boxprops=dict(facecolor='white', edgecolor=GREY_DARK, linewidth=1.5, alpha=0.8),
                whiskerprops=dict(color=GREY_DARK, linewidth=1.5, linestyle='-'),
                capprops=dict(color=GREY_DARK, linewidth=1.5))

means = [df[df['test_name'] == test]['z_score'].mean() for test in tests]
ax.scatter(range(len(tests)), means, color='black', s=100, zorder=3, 
          marker='D', edgecolors='white', linewidths=1.5)

for i, test in enumerate(tests):
    test_data = df[df['test_name'] == test]
    test_data_imp = isolated_imp_df[isolated_imp_df['test_name'] == test]
    test_data_over = isolated_over_df[isolated_over_df['test_name'] == test]
    
    if len(test_data_imp) > 0:
        np.random.seed(42)
        jitter = np.random.normal(0, 0.08, size=len(test_data_imp))
        ax.scatter([i] * len(test_data_imp) + jitter, 
                  test_data_imp['z_score'].values,
                  color=RED, s=60, alpha=0.7, edgecolors='darkred', 
                  linewidths=1.5, zorder=5, marker='o')
    
    if len(test_data_over) > 0:
        np.random.seed(43)
        jitter = np.random.normal(0, 0.08, size=len(test_data_over))
        ax.scatter([i] * len(test_data_over) + jitter, 
                  test_data_over['z_score'].values,
                  color=BLUE, s=60, alpha=0.7, edgecolors='darkblue', 
                  linewidths=1.5, zorder=5, marker='o')

#Create horizontal grid reference lines
ax.axhline(0, color='black', linestyle='--', linewidth=2, alpha=0.9, zorder=1)
ax.axhline(3, color=GREY_MED, linestyle=':', linewidth=1.5, alpha=0.8, zorder=1)
ax.axhline(-3, color=GREY_MED, linestyle=':', linewidth=1.5, alpha=0.8, zorder=1)
ax.axhline(1.5, color=GREY_MED, linestyle=':', linewidth=1, alpha=0.8, zorder=1)
ax.axhline(-1.5, color=GREY_MED, linestyle=':', linewidth=1, alpha=0.8, zorder=1)

#Title and axis labels
ax.set_title('Cognitive Test Z-Score Distributions with Isolated Extreme Cases Highlighted', 
            fontsize=25, fontweight='bold', color=GREY_DARK, pad=14)
ax.set_xlabel('Test Name', fontsize=20, color=GREY_DARK, fontweight='bold')
ax.set_ylabel('Z-Score (Standard Deviations from Expected)', fontsize=18, 
             color=GREY_DARK, fontweight='bold')
ax.set_xticks(range(len(tests)))
ax.set_xticklabels(tests, rotation=45, ha='right', fontsize=18)
ax.tick_params(axis='y', labelsize=18)
ax.set_yticks([-6, -3, -1.5, 0, 1.5, 3, 6])
ax.set_ylim(ax.get_ylim()[0], ax.get_ylim()[1] * 1.08)

ax.set_axisbelow(True)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color(GREY_MED)
ax.spines['bottom'].set_color(GREY_MED)
ax.spines['left'].set_linewidth(1.5)
ax.spines['bottom'].set_linewidth(1.5)

# LEGEND MOVED OUTSIDE (bbox_to_anchor places it outside the plot area)
legend_elements = [
    plt.Line2D([0], [0], color='black', linestyle='--', linewidth=2, label='Expected Mean (0)'),
    plt.Line2D([0], [0], color=GREY_MED, linestyle=':', linewidth=1.5, label='±3 SD threshold'),
    plt.Line2D([0], [0], color=GREY_MED, linestyle=':', linewidth=1, label='±1.5 SD'),
    plt.scatter([0], [0], color='black', s=100, marker='D', edgecolors='white', linewidths=1.5, label='Mean'),
    plt.Line2D([0], [0], color='black', linewidth=2.5, label='Median'),
    plt.Rectangle((0, 0), 1, 1, facecolor='white', edgecolor=GREY_DARK, linewidth=1.5, label='IQR (25th-75th %ile)'),
    plt.scatter([0], [0], color=RED, s=60, alpha=0.7, edgecolors='darkred', linewidths=1.5, label='Isolated Impairment'),
    plt.scatter([0], [0], color=BLUE, s=60, alpha=0.7, edgecolors='darkblue', linewidths=1.5, label='Isolated Overachievement')
]
ax.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(.98, .98), 
         frameon=False, fancybox=False, edgecolor=GREY_LIGHT, fontsize=18, framealpha=0.95)

plt.tight_layout()
plt.savefig('z_score_enhanced_violin_with_isolated_poster.svg', dpi=800, bbox_inches='tight', facecolor='white', format='svg')
plt.show()

print("\nEnhanced violin plot saved as: z_score_enhanced_violin_with_isolated.png")